In [1]:
# conda create -n test_Phot python=3.11 -y
# conda activate test_Phot
# pip install astropy photutils numpy matplotlib ipykernel

In [ ]:
import os
import glob
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# --- CONFIG ---
DATA_DIR = r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts"
FWHM = 3.0          # estimated stellar FWHM in pixels — tune to your plate scale
THRESHOLD_SIGMA = 5.0  # detection threshold above background, in sigma
SAVE_PLOTS = True    # save PNGs showing detected stars
OUTPUT_DIR = os.path.join(DATA_DIR, "detections")

def detect_stars_in_file(filepath):
    with fits.open(filepath) as hdul:
        # Try to find the first HDU with actual image data
        data = None
        for hdu in hdul:
            if hdu.data is not None:
                data = hdu.data.astype(float)
                break
        if data is None:
            print(f"  No image data found in {filepath}")
            return None, None

    # Handle NaNs
    data = np.nan_to_num(data, nan=np.nanmedian(data))

    # Estimate background stats
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)

    # Run star finder
    daofind = DAOStarFinder(fwhm=FWHM, threshold=THRESHOLD_SIGMA * std)
    sources = daofind(data - median)

    return data, sources

def main():
    fits_files = glob.glob(os.path.join(DATA_DIR, "*.fits")) + \
                 glob.glob(os.path.join(DATA_DIR, "*.fit"))

    if not fits_files:
        print(f"No .fits files found in {DATA_DIR}")
        return

    if SAVE_PLOTS:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Found {len(fits_files)} FITS files.\n")

    for filepath in fits_files:
        fname = os.path.basename(filepath)
        print(f"Processing {fname} ...")

        try:
            data, sources = detect_stars_in_file(filepath)
        except Exception as e:
            print(f"  ERROR reading file: {e}")
            continue

        if sources is None:
            continue

        n_stars = len(sources) if sources is not None else 0
        print(f"  Detected {n_stars} sources")

        if n_stars > 0:
            # Print a quick preview of brightest sources
            sources.sort("flux", reverse=True)
            print(sources[:5]["xcentroid", "ycentroid", "flux"])

        if SAVE_PLOTS and n_stars > 0:
            fig, ax = plt.subplots(figsize=(8, 8))
            vmin, vmax = np.percentile(data, [5, 99])
            ax.imshow(data, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
            for x, y in zip(sources["xcentroid"], sources["ycentroid"]):
                ax.add_patch(Circle((x, y), radius=6, edgecolor="red",
                                     facecolor="none", linewidth=1))
            ax.set_title(f"{fname} — {n_stars} detections")
            out_path = os.path.join(OUTPUT_DIR, fname.replace(".fits", "_detected.png"))
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)

        print()

    print("Done.")

if __name__ == "__main__":
    main()